<a href="https://colab.research.google.com/github/Of-Calls/sisicallcall-verification-finetuning/blob/main/titanet_finetune_colab_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TitaNet-Small / TitaNet-Large Fine-tuning Colab Notebook v3

이 노트북은 지금까지 발생한 문제들을 반영한 안정화 버전입니다.

반영 사항:
- `/content` 전체 탐색 금지. ZIP은 `/content/titanet_extract` 전용 폴더에만 해제합니다.
- `pytorch_lightning` 금지. NeMo 2.x 기준으로 `lightning.pytorch`만 사용합니다.
- JSONL을 일반 JSON처럼 읽지 않습니다. 깨진 JSONL도 복구 가능한 robust reader를 포함합니다.
- `valid_manifest.json`은 학습 validation에 사용하지 않습니다.
- `train_manifest.json` 내부에서 `train_ft_manifest.json` / `val_ft_manifest.json`을 다시 나눕니다.
- `train_ft`와 `val_ft`는 같은 speaker label universe를 공유합니다.
- TitaNet decoder의 output class 수를 train speaker 수에 맞게 재설정합니다.
- 가중치와 결과는 원본 ZIP과 같은 Drive 폴더 아래 `titanet_finetune_runs/{RUN_ID}`에 저장합니다.

처음에는 `RUN_TITANET_SMALL=True`, `RUN_TITANET_LARGE=False`, `DRY_RUN=True`로 시작하세요.


In [ ]:

from pathlib import Path
from datetime import datetime

# 네가 알려준 ZIP 경로
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_small.zip")

# 학습 결과 저장 위치: ZIP과 같은 폴더 아래
DRIVE_SAVE_ROOT = DRIVE_ZIP_PATH.parent / "titanet_finetune_runs"
RUN_ID = datetime.now().strftime("titanet_%Y%m%d_%H%M%S")

DRY_RUN = False
LIMIT_TRAIN_ROWS = 0
LIMIT_VALID_ROWS = 0

RUN_TITANET_SMALL = True
RUN_TITANET_LARGE = False

MAX_EPOCHS = 3
LIMIT_TRAIN_BATCHES = 1.0
LIMIT_VAL_BATCHES = 1.0
BATCH_SIZE = 32
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
PRECISION = "16-mixed"
SEED = 42

print("DRIVE_ZIP_PATH:", DRIVE_ZIP_PATH)
print("DRIVE_SAVE_ROOT:", DRIVE_SAVE_ROOT)
print("RUN_ID:", RUN_ID)
print("DRY_RUN:", DRY_RUN)


DRIVE_ZIP_PATH: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_small.zip
DRIVE_SAVE_ROOT: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs
RUN_ID: titanet_20260427_032738
DRY_RUN: False


## Optional install / repair

런타임이 이미 정상이라면 아래 셀은 건너뛰어도 됩니다.

만약 `numpy._core.umath` 관련 ImportError가 나면:
1. `INSTALL_OR_REPAIR=True`로 바꿔 실행
2. 런타임 재시작
3. 처음부터 다시 실행


In [ ]:

INSTALL_OR_REPAIR = False

if INSTALL_OR_REPAIR:
    !pip install -q "nemo_toolkit[asr]"
    !pip uninstall -y numpy
    !pip install --no-cache-dir --force-reinstall "numpy==1.26.4"
    print("설치/복구 완료. 런타임을 재시작한 뒤 위에서부터 다시 실행하세요.")
else:
    print("Skip install/repair.")


Skip install/repair.


In [ ]:

import os
import sys
import json
import random
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import soundfile as sf

# NeMo 2.x 모델은 lightning.pytorch.LightningModule을 상속합니다.
# pytorch_lightning과 섞으면 TypeError가 납니다.
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

from omegaconf import OmegaConf, open_dict
from tqdm.auto import tqdm

print("python:", sys.version)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("lightning.pytorch:", pl.__version__)

pl.seed_everything(SEED, workers=True)


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy: 1.26.4
torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-40GB
lightning.pytorch: 2.4.0


42

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

WORK_ZIP = Path("/content/titanet_local_subset_small.zip")
EXTRACT_ROOT = Path("/content/titanet_extract")

assert DRIVE_ZIP_PATH.exists(), f"ZIP not found: {DRIVE_ZIP_PATH}"

# 전용 extract 폴더만 정리합니다. /content 전체 탐색/정리 금지.
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not WORK_ZIP.exists() or WORK_ZIP.stat().st_size != DRIVE_ZIP_PATH.stat().st_size:
    print("Copy ZIP to /content ...")
    shutil.copy2(DRIVE_ZIP_PATH, WORK_ZIP)
else:
    print("ZIP already copied:", WORK_ZIP)

with zipfile.ZipFile(WORK_ZIP, "r") as zf:
    names = zf.namelist()
    print("ZIP first 30 entries:")
    for n in names[:30]:
        print(" -", n)
    print("Extract to:", EXTRACT_ROOT)
    zf.extractall(EXTRACT_ROOT)

# /content 전체가 아니라 EXTRACT_ROOT 아래만 탐색
candidates = []
if (EXTRACT_ROOT / "manifests").is_dir() and (EXTRACT_ROOT / "wavs").is_dir():
    candidates.append(EXTRACT_ROOT)

for p in EXTRACT_ROOT.rglob("*"):
    if not p.is_dir():
        continue
    try:
        if (p / "manifests").is_dir() and (p / "wavs").is_dir():
            candidates.append(p)
    except OSError:
        continue

candidates = sorted(set(candidates), key=lambda x: len(str(x)))
print("Detected candidates:")
for c in candidates:
    print(" -", c)

if not candidates:
    print("Extract root children:")
    for p in EXTRACT_ROOT.iterdir():
        print(" -", p)
    raise FileNotFoundError("Could not find dataset folder containing manifests/ and wavs/ inside EXTRACT_ROOT.")

DATA_DIR = candidates[0]
MANIFEST_DIR = DATA_DIR / "manifests"
REPORT_DIR = DATA_DIR / "reports"
REPORT_DIR.mkdir(exist_ok=True)

DRIVE_SAVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = DRIVE_SAVE_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("RUN_DIR:", RUN_DIR)
print("Manifest files:", [p.name for p in MANIFEST_DIR.iterdir()][:30])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP already copied: /content/titanet_local_subset_small.zip
ZIP first 30 entries:
 - manifests/
 - manifests/enrollment_manifest_local.csv
 - manifests/eval_manifest.json
 - manifests/evaluation_verification_manifest_local.csv
 - manifests/train_manifest.json
 - manifests/trials_local.csv
 - manifests/valid_manifest.json
 - README_TITANET_LOCAL_SUBSET.md
 - reports/
 - reports/audio_check_report.csv
 - reports/export_summary.json
 - reports/export_titanet_local_subset.log
 - reports/extracted_files.csv
 - reports/failed_extracts.csv
 - wavs/
 - wavs/eval/
 - wavs/eval/0000/
 - wavs/eval/0000/A0002-0000F1012-10120120-00000096_d2a8c8348e.wav
 - wavs/eval/0000/A0009-0000F1012-10120120-00367759_2a12669044.wav
 - wavs/eval/0000/A0010-0000F1012-10120120-00368091_2126cad9a4.wav
 - wavs/eval/0000/A0013-0000F1012-10120120-00368770_e6cd8d4fc3.wav
 - wavs/eval/0000/A001

In [ ]:

def read_json_objects_robust(path):
    """Reads normal JSONL and accidentally concatenated JSON objects."""
    path = Path(path)
    text = path.read_text(encoding="utf-8").strip()
    rows = []
    if not text:
        return rows

    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)

    while idx < n:
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break
        try:
            obj, next_idx = decoder.raw_decode(text, idx)
            rows.append(obj)
            idx = next_idx
        except json.JSONDecodeError as e:
            print(f"[JSON decode failed] path={path}, index={idx}")
            print("near text:", text[idx:idx+500])
            raise e
    return rows


def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
        return rows
    except json.JSONDecodeError:
        print(f"[WARN] Strict JSONL failed. Trying robust recovery: {path}")
        rows = read_json_objects_robust(path)
        if limit is not None:
            rows = rows[:limit]
        return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


print("JSONL utilities ready.")


JSONL utilities ready.


In [ ]:

COLAB_MANIFEST_DIR = DATA_DIR / "manifests_colab"
COLAB_MANIFEST_DIR.mkdir(exist_ok=True)

LOCAL_ROOT_MARKERS = [
    "D:/aihub_check/titanet_local_subset_small",
    r"D:\aihub_check\titanet_local_subset_small",
]

COLAB_ROOT_STR = str(DATA_DIR).replace("\\", "/")


def rewrite_path_to_colab(path_str):
    if path_str is None:
        return path_str
    s = str(path_str).replace("\\", "/")

    for marker in LOCAL_ROOT_MARKERS:
        m = marker.replace("\\", "/")
        if s.startswith(m):
            return s.replace(m, COLAB_ROOT_STR, 1)

    if s.startswith("/content/"):
        return s

    idx = s.find("/wavs/")
    if idx >= 0:
        return COLAB_ROOT_STR + s[idx:]

    return s


def rewrite_jsonl_manifest(src, dst):
    rows = read_jsonl(src)
    missing = 0
    for row in rows:
        row["audio_filepath"] = rewrite_path_to_colab(row.get("audio_filepath"))
        if not Path(row["audio_filepath"]).exists():
            missing += 1
    write_jsonl(dst, rows)
    return {"src": str(src), "dst": str(dst), "rows": len(rows), "missing_paths": missing}


def rewrite_audio_refs_field(x):
    if pd.isna(x):
        return x
    s = str(x)
    try:
        import ast
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            return str([rewrite_path_to_colab(v) for v in obj])
    except Exception:
        pass
    if ";" in s:
        return ";".join(rewrite_path_to_colab(v) for v in s.split(";"))
    return rewrite_path_to_colab(s)


def rewrite_csv(src, dst):
    df = pd.read_csv(src)
    for col in df.columns:
        if col in ["audio_ref", "local_audio_path", "verification_audio_ref"]:
            df[col] = df[col].map(rewrite_path_to_colab)
        elif col == "enrollment_audio_refs":
            df[col] = df[col].map(rewrite_audio_refs_field)
    df.to_csv(dst, index=False)
    return {"src": str(src), "dst": str(dst), "rows": len(df)}


rewrite_reports = []

for src_name, dst_name in [
    ("train_manifest.json", "train_manifest_colab.json"),
    ("valid_manifest.json", "valid_manifest_colab.json"),
    ("eval_manifest.json", "eval_manifest_colab.json"),
]:
    rewrite_reports.append(rewrite_jsonl_manifest(MANIFEST_DIR / src_name, COLAB_MANIFEST_DIR / dst_name))

for src_name, dst_name in [
    ("trials_local.csv", "trials_colab.csv"),
    ("enrollment_manifest_local.csv", "enrollment_manifest_colab.csv"),
    ("evaluation_verification_manifest_local.csv", "evaluation_verification_manifest_colab.csv"),
]:
    src = MANIFEST_DIR / src_name
    if src.exists():
        rewrite_reports.append(rewrite_csv(src, COLAB_MANIFEST_DIR / dst_name))

report_path = RUN_DIR / "colab_path_rewrite_report.json"
report_path.write_text(json.dumps(rewrite_reports, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(rewrite_reports, ensure_ascii=False, indent=2))

for r in rewrite_reports:
    if r.get("missing_paths", 0) > 0:
        raise FileNotFoundError(f"Missing paths after rewrite: {r}")


[
  {
    "src": "/content/titanet_extract/manifests/train_manifest.json",
    "dst": "/content/titanet_extract/manifests_colab/train_manifest_colab.json",
    "rows": 14725,
    "missing_paths": 0
  },
  {
    "src": "/content/titanet_extract/manifests/valid_manifest.json",
    "dst": "/content/titanet_extract/manifests_colab/valid_manifest_colab.json",
    "rows": 8400,
    "missing_paths": 0
  },
  {
    "src": "/content/titanet_extract/manifests/eval_manifest.json",
    "dst": "/content/titanet_extract/manifests_colab/eval_manifest_colab.json",
    "rows": 3000,
    "missing_paths": 0
  },
  {
    "src": "/content/titanet_extract/manifests/trials_local.csv",
    "dst": "/content/titanet_extract/manifests_colab/trials_colab.csv",
    "rows": 30000
  },
  {
    "src": "/content/titanet_extract/manifests/enrollment_manifest_local.csv",
    "dst": "/content/titanet_extract/manifests_colab/enrollment_manifest_colab.csv",
    "rows": 757
  },
  {
    "src": "/content/titanet_extract/mani

In [ ]:

SOURCE_TRAIN_MANIFEST = COLAB_MANIFEST_DIR / "train_manifest_colab.json"
assert SOURCE_TRAIN_MANIFEST.exists(), SOURCE_TRAIN_MANIFEST

TRAIN_FT_MANIFEST = COLAB_MANIFEST_DIR / "train_ft_manifest.json"
VAL_FT_MANIFEST = COLAB_MANIFEST_DIR / "val_ft_manifest.json"

rows = read_jsonl(SOURCE_TRAIN_MANIFEST)

# Dry-run limit with label-balanced round-robin sampling.
if LIMIT_TRAIN_ROWS and LIMIT_TRAIN_ROWS > 0:
    by_label_tmp = defaultdict(list)
    for r in rows:
        by_label_tmp[r["label"]].append(r)

    rng = random.Random(SEED)
    labels_tmp = sorted(by_label_tmp.keys())
    for lab in labels_tmp:
        rng.shuffle(by_label_tmp[lab])

    sampled = []
    exhausted = False
    while len(sampled) < LIMIT_TRAIN_ROWS and not exhausted:
        exhausted = True
        for lab in labels_tmp:
            if by_label_tmp[lab] and len(sampled) < LIMIT_TRAIN_ROWS:
                sampled.append(by_label_tmp[lab].pop())
                exhausted = False
    rows = sampled

rows_by_label = defaultdict(list)
for r in rows:
    rows_by_label[r["label"]].append(r)

train_rows = []
val_rows = []
rng = random.Random(SEED)

for label, lab_rows in rows_by_label.items():
    rng.shuffle(lab_rows)
    n = len(lab_rows)
    if n >= 10:
        n_val = max(1, int(n * 0.1))
    elif n >= 3:
        n_val = 1
    else:
        n_val = 0

    val_rows.extend(lab_rows[:n_val])
    train_rows.extend(lab_rows[n_val:])

write_jsonl(TRAIN_FT_MANIFEST, train_rows)
write_jsonl(VAL_FT_MANIFEST, val_rows)

# Re-read and rewrite to guarantee clean JSONL
train_rows = read_jsonl(TRAIN_FT_MANIFEST)
val_rows = read_jsonl(VAL_FT_MANIFEST)
write_jsonl(TRAIN_FT_MANIFEST, train_rows)
write_jsonl(VAL_FT_MANIFEST, val_rows)

train_labels = set(r["label"] for r in train_rows)
val_labels = set(r["label"] for r in val_rows)

summary = {
    "source_train_rows_after_limit": len(rows),
    "train_ft_rows": len(train_rows),
    "val_ft_rows": len(val_rows),
    "train_ft_speakers": len(train_labels),
    "val_ft_speakers": len(val_labels),
    "val_labels_not_in_train": len(val_labels - train_labels),
    "train_label_sample": Counter(r["label"] for r in train_rows).most_common(10),
    "val_label_sample": Counter(r["label"] for r in val_rows).most_common(10),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert len(val_labels - train_labels) == 0
assert len(train_rows) > 0
assert len(val_rows) > 0


{
  "source_train_rows_after_limit": 14725,
  "train_ft_rows": 13261,
  "val_ft_rows": 1464,
  "train_ft_speakers": 300,
  "val_ft_speakers": 300,
  "val_labels_not_in_train": 0,
  "train_label_sample": [
    [
      "0015",
      45
    ],
    [
      "0025",
      45
    ],
    [
      "0036",
      45
    ],
    [
      "0057",
      45
    ],
    [
      "0059",
      45
    ],
    [
      "0142",
      45
    ],
    [
      "0150",
      45
    ],
    [
      "0197",
      45
    ],
    [
      "0221",
      45
    ],
    [
      "0313",
      45
    ]
  ],
  "val_label_sample": [
    [
      "0015",
      5
    ],
    [
      "0025",
      5
    ],
    [
      "0036",
      5
    ],
    [
      "0057",
      5
    ],
    [
      "0059",
      5
    ],
    [
      "0142",
      5
    ],
    [
      "0150",
      5
    ],
    [
      "0197",
      5
    ],
    [
      "0221",
      5
    ],
    [
      "0313",
      5
    ]
  ]
}


In [ ]:

def validate_manifest_audio(manifest_path, sample_count=100):
    manifest_path = Path(manifest_path)
    rows = read_jsonl(manifest_path, limit=sample_count)
    ok = 0
    failed = 0
    failed_items = []

    for i, row in enumerate(tqdm(rows, desc=f"validate {manifest_path.name}")):
        audio_path = row.get("audio_filepath")
        label = row.get("label")
        try:
            if not audio_path:
                raise ValueError("missing audio_filepath")
            p = Path(audio_path)
            if not p.exists():
                raise FileNotFoundError(str(p))
            info = sf.info(str(p))
            if info.samplerate != 16000:
                raise ValueError(f"bad sample_rate={info.samplerate}")
            if info.channels != 1:
                raise ValueError(f"bad channels={info.channels}")
            if info.duration <= 0:
                raise ValueError(f"bad duration={info.duration}")
            if label is None or str(label) == "":
                raise ValueError("missing label")
            ok += 1
        except Exception as e:
            failed += 1
            failed_items.append({
                "index": i,
                "audio_filepath": audio_path,
                "label": label,
                "error": repr(e),
            })

    report = {
        "manifest": str(manifest_path),
        "checked": len(rows),
        "ok": ok,
        "failed": failed,
        "failed_items_sample": failed_items[:20],
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

val_reports = [
    validate_manifest_audio(TRAIN_FT_MANIFEST, 100),
    validate_manifest_audio(VAL_FT_MANIFEST, 100),
]
(RUN_DIR / "manifest_audio_validation_report.json").write_text(
    json.dumps(val_reports, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
assert all(r["failed"] == 0 for r in val_reports)


validate train_ft_manifest.json:   0%|          | 0/100 [00:00<?, ?it/s]

{
  "manifest": "/content/titanet_extract/manifests_colab/train_ft_manifest.json",
  "checked": 100,
  "ok": 100,
  "failed": 0,
  "failed_items_sample": []
}


validate val_ft_manifest.json:   0%|          | 0/100 [00:00<?, ?it/s]

{
  "manifest": "/content/titanet_extract/manifests_colab/val_ft_manifest.json",
  "checked": 100,
  "ok": 100,
  "failed": 0,
  "failed_items_sample": []
}


In [ ]:

from nemo.collections.asr.models import EncDecSpeakerLabelModel

tmp_model = EncDecSpeakerLabelModel.from_pretrained("titanet_small")
print("is lightning.pytorch LightningModule:", isinstance(tmp_model, pl.LightningModule))
print("MRO:")
for x in type(tmp_model).mro()[:12]:
    print(" -", x)

assert isinstance(tmp_model, pl.LightningModule), "Use lightning.pytorch, not pytorch_lightning."
del tmp_model
torch.cuda.empty_cache()


[NeMo I 2026-04-27 03:28:04 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
[NeMo I 2026-04-27 03:28:04 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo
[NeMo I 2026-04-27 03:28:04 common:939] Instantiating model from pre-trained checkpoint


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-04-27 03:28:10 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
is lightning.pytorch LightningModule: True
MRO:
 - <class 'nemo.collections.asr.models.label_models.EncDecSpeakerLabelModel'>
 - <class 'nemo.core.classes.modelPT.ModelPT'>
 - <class 'lightning.pytorch.core.module.LightningModule'>
 - <class 'lightning.fabric.utilities.device_dtype_mixin._DeviceDtypeModuleMixin'>
 - <class 'lightning.pytorch.core.mixins.hparams_mixin.HyperparametersMixin'>
 - <class 'lightning.pytorch.core.hooks.ModelHooks'>
 - <class 'lightning.pytorch.core.hooks.DataHooks'>
 - <class 'lightning.pytorch.core.hooks.CheckpointHooks'>
 - <class 'torch.nn.modules.module.Module'>
 - <class 'nemo.core.classes.common.Model'>
 - <class 'nemo.core.classes.common.Typing'>
 - <class 'nemo.core.classes.common.Serialization'>


In [ ]:
from torchmetrics.classification import MulticlassAccuracy
import torch.nn as nn

def get_train_labels(manifest_path):
    rows = read_jsonl(manifest_path)
    return sorted({str(r["label"]) for r in rows})


def replace_last_linear_in_module(module, out_features):
    linear_names = []
    for name, child in module.named_modules():
        if isinstance(child, nn.Linear):
            linear_names.append(name)

    if not linear_names:
        return False

    target_name = linear_names[-1]
    parent = module
    parts = target_name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    old_linear = getattr(parent, parts[-1])

    new_linear = nn.Linear(
        in_features=old_linear.in_features,
        out_features=out_features,
        bias=old_linear.bias is not None,
    )
    new_linear.to(device=old_linear.weight.device, dtype=old_linear.weight.dtype)
    setattr(parent, parts[-1], new_linear)

    print(f"Replaced decoder linear: {target_name}, {old_linear.out_features} -> {out_features}")
    return True

def reset_classification_metrics(model, num_classes):
    device = next(model.parameters()).device

    # NeMo speaker model에서 주로 쓰는 metric들 재초기화
    metric_names = [
        "_macro_accuracy",
        "_pair_macro_accuracy",
    ]

    for name in metric_names:
        if hasattr(model, name):
            try:
                setattr(
                    model,
                    name,
                    MulticlassAccuracy(
                        num_classes=num_classes,
                        average="macro"
                    ).to(device)
                )
                print(f"Reset metric: {name} -> num_classes={num_classes}")
            except Exception as e:
                print(f"[WARN] failed to reset {name}: {e}")

    # TopKClassificationAccuracy는 NeMo 커스텀일 수 있어서 가능하면 num_classes 속성만 갱신
    if hasattr(model, "_accuracy"):
        acc = getattr(model, "_accuracy")
        for attr in ["num_classes", "_num_classes", "n_classes"]:
            if hasattr(acc, attr):
                try:
                    setattr(acc, attr, num_classes)
                    print(f"Updated _accuracy.{attr} -> {num_classes}")
                except Exception as e:
                    print(f"[WARN] failed to update _accuracy.{attr}: {e}")

def configure_speaker_model_for_finetune(model, train_manifest, val_manifest, labels, batch_size=16, num_workers=2, lr=1e-4):
    num_classes = len(labels)

    with open_dict(model.cfg):
        model.cfg.labels = labels

        if "decoder" in model.cfg:
            for k in ["num_classes", "num_classes_total", "n_classes", "num_speakers"]:
                if k in model.cfg.decoder:
                    model.cfg.decoder[k] = num_classes

        if "optim" in model.cfg and model.cfg.optim is not None:
            model.cfg.optim.lr = lr

    for attr in ["labels", "_labels"]:
        try:
            setattr(model, attr, labels)
        except Exception:
            pass

    replaced = replace_last_linear_in_module(model.decoder, num_classes)
    if not replaced:
        print("[WARN] No nn.Linear found in model.decoder. Decoder mismatch may still occur.")

    reset_classification_metrics(model, num_classes)

    train_cfg = OmegaConf.create({
        "manifest_filepath": str(train_manifest),
        "sample_rate": 16000,
        "labels": labels,
        "batch_size": batch_size,
        "shuffle": True,
        "is_tarred": False,
        "num_workers": num_workers,
        "pin_memory": True,
    })

    val_cfg = OmegaConf.create({
        "manifest_filepath": str(val_manifest),
        "sample_rate": 16000,
        "labels": labels,
        "batch_size": batch_size,
        "shuffle": False,
        "is_tarred": False,
        "num_workers": num_workers,
        "pin_memory": True,
    })

    model.setup_training_data(train_cfg)
    model.setup_validation_data(val_cfg)
    return model


def make_trainer(model_name, run_dir):
    model_run_dir = Path(run_dir) / model_name
    model_run_dir.mkdir(parents=True, exist_ok=True)

    ckpt_cb = ModelCheckpoint(
        dirpath=str(model_run_dir / "checkpoints"),
        filename="{epoch}-{step}-{val_loss:.4f}",
        monitor="val_loss",
        mode="min",
        save_top_k=2,
        save_last=True,
    )

    lr_cb = LearningRateMonitor(logging_interval="step")
    logger = CSVLogger(save_dir=str(model_run_dir), name="logs")

    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision=PRECISION if torch.cuda.is_available() else "32-true",
        max_epochs=MAX_EPOCHS,
        limit_train_batches=LIMIT_TRAIN_BATCHES,
        limit_val_batches=LIMIT_VAL_BATCHES,
        log_every_n_steps=10,
        callbacks=[ckpt_cb, lr_cb],
        logger=logger,
        default_root_dir=str(model_run_dir),
        enable_checkpointing=True,
        num_sanity_val_steps=0,
    )
    return trainer, model_run_dir


print("Training helpers ready.")


Training helpers ready.


In [ ]:

def train_one_model(model_name):
    labels = get_train_labels(TRAIN_FT_MANIFEST)
    print("num train labels:", len(labels), labels[:10])

    model = EncDecSpeakerLabelModel.from_pretrained(model_name)
    print("Loaded:", model_name)
    print("LightningModule:", isinstance(model, pl.LightningModule))

    model = configure_speaker_model_for_finetune(
        model=model,
        train_manifest=TRAIN_FT_MANIFEST,
        val_manifest=VAL_FT_MANIFEST,
        labels=labels,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        lr=LEARNING_RATE,
    )

    trainer, model_run_dir = make_trainer(model_name, RUN_DIR)
    print("Start fine-tuning:", model_name)
    trainer.fit(model)

    nemo_path = model_run_dir / f"{model_name}_finetuned_final.nemo"
    model.save_to(str(nemo_path))

    (model_run_dir / "labels.json").write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8")
    OmegaConf.save(model.cfg, model_run_dir / "model_cfg.yaml")

    meta = {
        "model_name": model_name,
        "nemo_path": str(nemo_path),
        "model_run_dir": str(model_run_dir),
        "num_labels": len(labels),
        "train_manifest": str(TRAIN_FT_MANIFEST),
        "val_manifest": str(VAL_FT_MANIFEST),
        "dry_run": DRY_RUN,
    }
    (model_run_dir / "train_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", nemo_path)
    return model, meta


trained = {}

if RUN_TITANET_SMALL:
    model_small, meta_small = train_one_model("titanet_small")
    trained["titanet_small"] = meta_small
    print(json.dumps(meta_small, ensure_ascii=False, indent=2))
else:
    print("Skip titanet_small")

if RUN_TITANET_LARGE:
    model_large, meta_large = train_one_model("titanet_large")
    trained["titanet_large"] = meta_large
    print(json.dumps(meta_large, ensure_ascii=False, indent=2))
else:
    print("Skip titanet_large")

print("trained:", json.dumps(trained, ensure_ascii=False, indent=2))


num train labels: 300 ['0015', '0025', '0036', '0057', '0059', '0077', '0111', '0142', '0150', '0197']
[NeMo I 2026-04-27 03:28:10 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
[NeMo I 2026-04-27 03:28:10 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo
[NeMo I 2026-04-27 03:28:10 common:939] Instantiating model from pre-trained checkpoint


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-04-27 03:28:15 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
Loaded: titanet_small
LightningModule: True
Replaced decoder linear: final, 16681 -> 300
Reset metric: _macro_accuracy -> num_classes=300
Reset metric: _pair_macro_accuracy -> num_classes=300
[NeMo I 2026-04-27 03:28:16 collections:750] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2026-04-27 03:28:16 collections:751] Dataset successfully loaded with 13261 items and total duration provided from manifest is  7.39 hours.
[NeMo I 2026-04-27 03:28:16 collections:757] # 13261 files loaded accounting to # 300 labels


[NeMo W 2026-04-27 03:28:16 label_models:201] Total number of 300 labels found in all the manifest files.


[NeMo I 2026-04-27 03:28:16 collections:750] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2026-04-27 03:28:16 collections:751] Dataset successfully loaded with 13261 items and total duration provided from manifest is  7.39 hours.
[NeMo I 2026-04-27 03:28:16 collections:757] # 13261 files loaded accounting to # 300 labels
[NeMo I 2026-04-27 03:28:16 collections:750] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2026-04-27 03:28:16 collections:751] Dataset successfully loaded with 1464 items and total duration provided from manifest is  0.82 hours.
[NeMo I 2026-04-27 03:28:16 collections:757] # 1464 files loaded accounting to # 300 labels


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


Saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo
{
  "model_name": "titanet_small",
  "nemo_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo",
  "model_run_dir": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small",
  "num_labels": 300,
  "train_manifest": "/content/titanet_extract/manifests_colab/train_ft_manifest.json",
  "val_manifest": "/content/titanet_extract/manifests_colab/val_ft_manifest.json",
  "dry_run": false
}
Skip titanet_large
trained: {
  "titanet_small": {
    "model_name": "titanet_small",
    "nemo_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_smal

## After training

학습이 끝나면 아래를 확인하세요.

- `.nemo` 저장 위치: `DRIVE_SAVE_ROOT / RUN_ID / model_name / {model_name}_finetuned_final.nemo`
- labels: `labels.json`
- NeMo config: `model_cfg.yaml`
- Lightning checkpoints: `checkpoints/`

처음 dry-run이 성공하면:
1. `DRY_RUN=False`
2. `RUN_TITANET_SMALL=True`
3. `RUN_TITANET_LARGE=False`
4. `MAX_EPOCHS=1~3`
로 small부터 본 실험을 진행하세요.

Large는 small이 끝까지 성공한 뒤 켜세요.


In [ ]:

print("RUN_DIR:", RUN_DIR)
if RUN_DIR.exists():
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file():
            print(p)


RUN_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/colab_path_rewrite_report.json
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/manifest_audio_validation_report.json
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/checkpoints/epoch=1-step=830-val_loss=10.1745.ckpt
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/checkpoints/epoch=2-step=1245-val_loss=10.0709.ckpt
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/checkpoints/last.ckpt
/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls

In [ ]:
metrics_files = list(Path(RUN_DIR).rglob("metrics.csv"))
print(metrics_files)

for mf in metrics_files:
    print("metrics:", mf)
    df = pd.read_csv(mf)
    display(df.tail(20))
    print(df.columns.tolist())

[PosixPath('/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/logs/version_0/metrics.csv')]
metrics: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/logs/version_0/metrics.csv


,epoch,global_step,learning_rate,loss,lr-SGD,step,training_batch_accuracy_top_0,val_acc_macro,val_acc_micro_top_1,val_loss
231,2.0,1149.0,1.798663e-06,10.437703,NaN,1149,0.06250,NaN,NaN,NaN
232,NaN,NaN,NaN,NaN,1.445181e-06,1159,NaN,NaN,NaN,NaN
233,2.0,1159.0,1.445181e-06,10.336410,NaN,1159,0.15625,NaN,NaN,NaN
234,NaN,NaN,NaN,NaN,1.129831e-06,1169,NaN,NaN,NaN,NaN
235,2.0,1169.0,1.129830e-06,10.040032,NaN,1169,0.25000,NaN,NaN,NaN
236,NaN,NaN,NaN,NaN,8.528603e-07,1179,NaN,NaN,NaN,NaN
237,2.0,1179.0,8.528602e-07,10.004238,NaN,1179,0.21875,NaN,NaN,NaN
238,NaN,NaN,NaN,NaN,6.144874e-07,1189,NaN,NaN,NaN,NaN
239,2.0,1189.0,6.144874e-07,10.079201,NaN,1189,0.21875,NaN,NaN,NaN
240,NaN,NaN,NaN,NaN,4.148992e-07,1199,NaN,NaN,NaN,NaN


['epoch', 'global_step', 'learning_rate', 'loss', 'lr-SGD', 'step', 'training_batch_accuracy_top_0', 'val_acc_macro', 'val_acc_micro_top_1', 'val_loss']


In [ ]:
from nemo.collections.asr.models import EncDecSpeakerLabelModel
from pathlib import Path

nemo_path = Path(RUN_DIR) / "titanet_small" / "titanet_small_finetuned_final.nemo"
print(nemo_path, nemo_path.exists())

reloaded = EncDecSpeakerLabelModel.restore_from(str(nemo_path))
print("reloaded OK:", type(reloaded))

/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo True


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-04-27 03:38:31 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_20260427_032738/titanet_small/titanet_small_finetuned_final.nemo.
reloaded OK: <class 'nemo.collections.asr.models.label_models.EncDecSpeakerLabelModel'>
